# 🎯 Week 4 Lab: Multi-Joint Reaching, Muscle Synergies & PCA
## From EMG Signals to Motor Module Analysis

**Course:** Machine Learning for Movement Science
**Topics:** 2-DOF arm model · EMG simulation · PCA dimensionality reduction · Varimax rotation · Clinical interpretation
**Dataset:** 20 subjects (10 healthy, 10 impaired) × 8 targets × 3 speeds = 480 trials
**Reference:** Clark et al. (2010) *J Neurophysiol* — Motor module merging post-stroke

### Learning Objectives
By the end of this lab you will be able to:
1. Compute forward kinematics for a 2-DOF planar arm
2. Generate and process simulated EMG data (rectification, envelope, peak amplitude)
3. Apply PCA (via sklearn) to identify muscle synergies from multi-muscle datasets
4. Interpret scree plots, loading heatmaps, and PC scores in a clinical context
5. Apply Varimax rotation and explain why it aids physiological interpretation
6. Compare healthy vs. impaired motor module structure

### Difficulty Legend
- 🟢 **Green:** Provided code — run and understand
- 🟡 **Yellow:** Fill in 1–3 lines
- 🟠 **Orange:** Write a function or short block
- 🔴 **Red:** Multi-step challenge

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# Color palette
NAVY = '#1B3A5C'; TEAL = '#2E8B8B'; BLUE = '#3498db'
RED = '#e74c3c'; GREEN = '#2ecc71'; ORANGE = '#e67e22'
PURPLE = '#9b59b6'; GRAY = '#7f8c8d'

---
## 🟡 Part 1: 2-DOF Arm Model & Forward Kinematics

### Exercise 1.1: Forward Kinematics
Complete the function to compute hand (x, y) from joint angles.

Recall from lecture:
$$x = L_1 \cos(q_1) + L_2 \cos(q_1 + q_2)$$
$$y = L_1 \sin(q_1) + L_2 \sin(q_1 + q_2)$$

In [ ]:
# Exercise 1.1: Forward kinematics
L1 = 0.30  # upper arm (m)
L2 = 0.35  # forearm (m)

def forward_kinematics(q1, q2):
    """Compute hand (x, y) from joint angles q1, q2 (radians)."""
    ### YOUR CODE HERE (2 lines) ###
    x = None
    y = None
    ### END YOUR CODE ###
    return x, y

# Test
q1_rest = np.deg2rad(45)
q2_rest = np.deg2rad(90)
x0, y0 = forward_kinematics(q1_rest, q2_rest)
print(f"Rest hand position: ({x0:.3f}, {y0:.3f}) m")
# Expected: approximately (-0.035, 0.460)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Use `np.cos()` and `np.sin()`. Remember `q1 + q2` gives the absolute elbow angle.

```python
x = L1 * np.cos(q1) + L2 * np.cos(q1 + q2)
```
</details>

### 🟢 Exercise 1.2: Visualize arm and targets
This code is provided — run it to see the setup.

In [ ]:
# Exercise 1.2 (provided): Arm visualization
n_targets = 8
reach_dist = 0.10
target_angles = np.linspace(0, 2 * np.pi, n_targets, endpoint=False)
targets_x = x0 + reach_dist * np.cos(target_angles)
targets_y = y0 + reach_dist * np.sin(target_angles)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
shoulder = (0, 0)
elbow = (L1 * np.cos(q1_rest), L1 * np.sin(q1_rest))
hand = (x0, y0)
ax.plot([shoulder[0], elbow[0]], [shoulder[1], elbow[1]], 'o-', color=NAVY, lw=5, ms=10)
ax.plot([elbow[0], hand[0]], [elbow[1], hand[1]], 'o-', color=TEAL, lw=5, ms=10)
ax.plot(*hand, 'o', color=RED, ms=12, zorder=6)
for i in range(n_targets):
    ax.plot(targets_x[i], targets_y[i], 's', color=ORANGE, ms=10)
    ax.annotate(f'{int(np.rad2deg(target_angles[i]))}°',
                xy=(targets_x[i], targets_y[i]), fontsize=8,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlim(-0.1, 0.5); ax.set_ylim(-0.05, 0.55)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('2-DOF Arm with Center-Out Targets', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🟢 Part 2: EMG Simulation — Synergy-Based Model

### Exercise 2.1: Understand the synergy model
Run this cell and study the weight matrices. Each row is a synergy; each column is a muscle.
- **Healthy:** 3 synergies (extensors, flexors, elbow modulation)
- **Impaired:** 2 synergies (3rd merged into first two → broader co-contraction)

In [ ]:
# Exercise 2.1 (provided): Synergy weight matrices
muscle_names = ['Sh.H.Flex', 'Sh.H.Ext', 'El.Flex', 'El.Ext', 'Bi.Flex', 'Bi.Ext']

W_healthy = np.array([
    [0.05, 0.80, 0.05, 0.75, 0.05, 0.85],  # Syn1: extensors (PD ~30°)
    [0.80, 0.05, 0.55, 0.05, 0.85, 0.05],  # Syn2: flexors (PD ~210°)
    [0.15, 0.55, 0.85, 0.10, 0.10, 0.50],  # Syn3: elbow modulation (PD ~100°)
])
synergy_pds_healthy = np.deg2rad([30, 210, 100])

W_impaired = np.array([
    [0.20, 0.80, 0.35, 0.75, 0.30, 0.85],  # Merged ext + elbow
    [0.80, 0.20, 0.75, 0.35, 0.85, 0.30],  # Merged flex + elbow
])
synergy_pds_impaired = np.deg2rad([45, 225])

print(f"Healthy: {W_healthy.shape[0]} synergies × {W_healthy.shape[1]} muscles")
print(f"Impaired: {W_impaired.shape[0]} synergies × {W_impaired.shape[1]} muscles")
print(f"\nQ: What muscles does Synergy 3 (healthy) recruit most strongly?")
print(f"   Look at row 3 of W_healthy: {W_healthy[2]}")

### 🟢 Exercise 2.2: EMG generation functions
Run this cell — it defines the EMG generation pipeline from lecture.

In [ ]:
# Exercise 2.2 (provided): EMG generation pipeline
dt = 0.001; T = 0.8; t = np.arange(0, T, dt)
speed_durations = [0.7, 0.5, 0.3]

def min_jerk(t_arr, dur):
    tau = np.clip(t_arr / dur, 0, 1)
    return 10 * tau**3 - 15 * tau**4 + 6 * tau**5

def narrow_tuning(angle, pd, width_deg=140):
    diff = np.arctan2(np.sin(angle - pd), np.cos(angle - pd))
    width = np.deg2rad(width_deg)
    if abs(diff) < width / 2:
        return np.cos(np.pi * diff / width)
    return 0.0

def lowpass_envelope(signal, window=40):
    kernel = np.ones(window) / window
    return np.convolve(signal, kernel, mode='same')

def generate_emg(target_angle, duration, noise_std=0.05, impaired=False):
    n_muscles = 6
    emg_clean = np.zeros((len(t), n_muscles))
    vel_profile = np.gradient(min_jerk(t, duration)) / dt
    vel_profile = vel_profile / (vel_profile.max() + 1e-8)
    W = W_impaired if impaired else W_healthy
    pds = synergy_pds_impaired if impaired else synergy_pds_healthy
    ns = noise_std * (2.5 if impaired else 1.0)
    for si in range(W.shape[0]):
        if not impaired:
            c_i = narrow_tuning(target_angle, pds[si], width_deg=140)
        else:
            c_i = max(np.cos(target_angle - pds[si]), 0.0)
        for mi in range(n_muscles):
            emg_clean[:, mi] += W[si, mi] * c_i * vel_profile
    raw_emg = emg_clean * (1.0 + np.random.normal(0, 0.6, emg_clean.shape))
    raw_emg += np.random.normal(0, ns * 0.3, emg_clean.shape)
    raw_emg = np.abs(raw_emg)
    envelope = np.zeros_like(raw_emg)
    for mi in range(n_muscles):
        envelope[:, mi] = lowpass_envelope(raw_emg[:, mi])
    return raw_emg, envelope

print("EMG functions defined ✓")

### 🟠 Exercise 2.3: Plot EMG for two opposite targets
Create a 6×2 subplot grid showing rectified EMG (light) with envelope overlay (dark) for targets at 45° and 225°.

In [ ]:
# Exercise 2.3: Plot EMG for 45° and 225°
fig, axes = plt.subplots(6, 2, figsize=(12, 10), sharex=True)

for col, (angle, label) in enumerate([(np.deg2rad(45), '45° (reach right)'),
                                       (np.deg2rad(225), '225° (pull left)')]):
    raw, env = generate_emg(angle, 0.5)
    for mi in range(6):
        ax = axes[mi, col]
        ### YOUR CODE HERE (2 lines): plot raw (light, thin) and envelope (dark, thick) ###
        
        
        ### END YOUR CODE ###
        if col == 0:
            ax.set_ylabel(muscle_names[mi], fontsize=9)
        if mi == 0:
            ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_ylim(bottom=0)

axes[-1, 0].set_xlabel('Time (ms)'); axes[-1, 1].set_xlabel('Time (ms)')
fig.suptitle('EMG Traces: Reciprocal Activation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
ax.plot(t * 1000, raw[:, mi], color=BLUE, alpha=0.3, lw=0.5)
ax.plot(t * 1000, env[:, mi], color=NAVY, lw=2)
```
</details>

---
## 🟡 Part 3: Build the Full Dataset

### Exercise 3.1: Generate 480 trials
Complete the loop: for each trial, generate EMG, extract peak envelope, store it.

In [ ]:
# Exercise 3.1: Generate the dataset
n_subjects = 20; n_healthy = 10
all_features = []; all_labels = []; all_targets = []; all_speeds = []

for si in range(n_subjects):
    is_imp = si >= n_healthy
    for ti, ta in enumerate(target_angles):
        for spi, dur in enumerate(speed_durations):
            ### YOUR CODE HERE (3 lines): generate EMG, extract peak envelope, append ###
            
            
            
            ### END YOUR CODE ###
            all_labels.append('impaired' if is_imp else 'healthy')
            all_targets.append(ti)
            all_speeds.append(spi)

X_raw = np.array(all_features)  # should be (480, 6)
labels = np.array(all_labels)
targets = np.array(all_targets)
speeds = np.array(all_speeds)

print(f"Dataset shape: {X_raw.shape}")
# Expected: (480, 6)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
_, env = generate_emg(ta, dur, impaired=is_imp)
peak = env.max(axis=0)
all_features.append(peak)
```
</details>

### 🟢 Exercise 3.1b: Reaching Trajectories — Healthy vs Impaired

Before we look at tuning curves, let's visualize what these reaches look like in space.
Plot hand trajectories for one healthy and one impaired subject reaching to
4 diagonal targets (45°, 135°, 225°, 315°), 5 trials each.

The impaired subject should show more curved, variable paths due to the merged synergy structure.


In [ ]:
# Exercise 3.1b: Reaching trajectories for 4 diagonal targets
diag_degs = [45, 135, 225, 315]
diag_angles = [np.deg2rad(a) for a in diag_degs]
diag_colors = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12']
n_trial_vis = 5

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax_idx, (title, is_imp) in enumerate([('Healthy Subject', False),
                                           ('Impaired Subject', True)]):
    ax = axes[ax_idx]
    np.random.seed(200 + ax_idx)
    
    for ai, (angle, deg) in enumerate(zip(diag_angles, diag_degs)):
        tx = x0 + reach_dist * np.cos(angle)
        ty = y0 + reach_dist * np.sin(angle)
        
        for trial in range(n_trial_vis):
            dur = 0.5
            mj = min_jerk(t, dur)
            
            ### YOUR CODE HERE ###
            # 1. Create noise: noise_scale = 0.012 if impaired, 0.003 if healthy
            # 2. Create temporally correlated noise:
            #    nx = np.cumsum(np.random.randn(len(t)) * 0.001) * noise_scale
            # 3. Create perpendicular curvature (impaired: curve_scale=0.008, healthy: 0.001)
            #    curve_mag = curve_scale * np.sin(np.pi * mj) * np.random.randn() * 1.5
            # 4. Compute hand_x = x0 + (tx-x0)*mj + noise + curvature
            # 5. Plot with ax.plot(hand_x, hand_y, ...)
            ### END YOUR CODE ###
        
        ax.plot(tx, ty, 'o', color=diag_colors[ai], ms=12, zorder=5,
                markeredgecolor='k', markeredgewidth=1.0)
    
    ax.plot(x0, y0, 'ko', ms=8, zorder=6)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(title, fontsize=13, fontweight='bold', color=NAVY)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

plt.suptitle('Reaching Trajectories: 4 Diagonal Targets × 5 Trials',
             fontsize=13, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout(); plt.show()


### 🟡 Exercise 3.1c: Comparing Healthy vs Impaired EMG

Compare the EMG envelopes of **Sh.H.Flex** and **El.Flex** for reaches to 45° and 225°,
overlaying healthy (blue) and impaired (red) on the same axes.

Look for co-contraction in the impaired subject — muscles activating when they shouldn't.


In [ ]:
# Exercise 3.1c: Healthy vs Impaired EMG comparison
compare_muscles = [0, 2]  # Sh.H.Flex, El.Flex
compare_angles = [np.deg2rad(45), np.deg2rad(225)]
compare_labels = ['45° (reach right)', '225° (pull left)']

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

for col, (angle, dir_label) in enumerate(zip(compare_angles, compare_labels)):
    np.random.seed(42)
    _, env_h = generate_emg(angle, 0.5, impaired=False)
    np.random.seed(42)
    _, env_i = generate_emg(angle, 0.5, impaired=True)
    
    for row, mi in enumerate(compare_muscles):
        ax = axes[row, col]
        ### YOUR CODE HERE ###
        # Plot env_h[:, mi] in blue (solid) and env_i[:, mi] in red (dashed)
        # Use fill_between for shading
        # Set ylabel to muscle_names[mi], title to dir_label
        ### END YOUR CODE ###

axes[1, 0].set_xlabel('Time (ms)'); axes[1, 1].set_xlabel('Time (ms)')
fig.suptitle('Healthy vs Impaired: Same Muscle, Same Direction',
             fontsize=13, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout(); plt.show()


### 🟢 Exercise 3.2: Tuning curves (provided)

In [ ]:
# Exercise 3.2 (provided): Tuning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, grp in zip(axes, ['healthy', 'impaired']):
    mask = labels == grp
    for mi in range(6):
        means = [X_raw[mask & (targets == ti), mi].mean() for ti in range(n_targets)]
        ax.plot(np.rad2deg(target_angles), means, 'o-', label=muscle_names[mi], lw=2)
    ax.set_xlabel('Target Direction (°)'); ax.set_ylabel('Peak Envelope')
    ax.set_title(f'{grp.title()} Tuning Curves', fontweight='bold')
    ax.legend(fontsize=7, ncol=2); ax.set_xticks(np.arange(0, 360, 45))
plt.tight_layout(); plt.show()

---
## 🟠 Part 4: PCA — Standardize, Fit, Inspect

### Exercise 4.1: Apply StandardScaler + PCA to each group
1. Split data by group (healthy / impaired)
2. Standardize each group separately
3. Inject the post-standardization noise (provided for you)
4. Fit PCA and print variance ratios

In [ ]:
# Exercise 4.1: PCA on healthy and impaired
healthy_mask = labels == 'healthy'
impaired_mask = labels == 'impaired'

# --- Healthy ---
### YOUR CODE HERE (2 lines): create StandardScaler, fit_transform healthy data ###


### END YOUR CODE ###

# Inject 3rd synergy noise (provided — preserves independent elbow module)
np.random.seed(42)
syn3_noise = np.random.normal(0, 1.5, size=X_h.shape[0])
syn3_loadings = np.array([-0.1, 0.1, 0.7, 0.7, 0.4, 0.4])
X_h += np.outer(syn3_noise, syn3_loadings)

### YOUR CODE HERE (2 lines): fit PCA, transform to get scores ###


### END YOUR CODE ###

# --- Impaired ---
scaler_i = StandardScaler()
X_i = scaler_i.fit_transform(X_raw[impaired_mask])

np.random.seed(77)
cocontract_noise = np.random.normal(0, 1.5, size=X_i.shape[0])
cocontract_pattern = np.array([0.2, 0.7, 0.3, 0.7, 0.2, 0.7])
X_i += np.outer(cocontract_noise, cocontract_pattern)

pca_i = PCA().fit(X_i)
scores_i = pca_i.transform(X_i)

print("Explained variance ratios:")
print(f"  Healthy:  {pca_h.explained_variance_ratio_[:4].round(3)}")
print(f"  Impaired: {pca_i.explained_variance_ratio_[:4].round(3)}")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
scaler_h = StandardScaler()
X_h = scaler_h.fit_transform(X_raw[healthy_mask])
# ... noise injection ...
pca_h = PCA().fit(X_h)
scores_h = pca_h.transform(X_h)
```
</details>

### 🟠 Exercise 4.2: Scree plot
Create a scree plot for each group: bar chart of individual variance + line for cumulative.
Add a 90% threshold line. How many PCs does each group need?

In [ ]:
# Exercise 4.2: Scree plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (pca_obj, label, color) in zip(axes,
        [(pca_h, 'Healthy', BLUE), (pca_i, 'Impaired', RED)]):
    var = pca_obj.explained_variance_ratio_ * 100
    cumvar = np.cumsum(var)
    pcs = np.arange(1, len(var) + 1)
    
    ### YOUR CODE HERE (3 lines): bar chart, cumulative line, 90% threshold ###
    
    
    
    ### END YOUR CODE ###
    
    ax.set_xlabel('Principal Component'); ax.set_ylabel('Variance (%)')
    ax.set_title(f'{label}', fontweight='bold')
    ax.legend(fontsize=8); ax.set_xticks(pcs); ax.set_ylim(0, 105)

fig.suptitle('Scree Plots: How Many PCs for 90% Variance?', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
ax.bar(pcs, var, color=color, alpha=0.7, label='Individual')
ax.plot(pcs, cumvar, 'ko-', ms=6, label='Cumulative')
ax.axhline(90, color=GRAY, ls='--', lw=1, label='90% threshold')
```
</details>

---
## 🟠 Part 5: Varimax Rotation

### Exercise 5.1: Implement Varimax
Varimax finds a rotation matrix $R$ that maximizes the variance of squared loadings.
The algorithm iterates: compute gradient → SVD → update $R$ until convergence.

In [ ]:
# Exercise 5.1: Varimax rotation
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """
    Varimax rotation of loading matrix Phi (n_variables × n_factors).
    Returns rotated loading matrix (same shape).
    """
    p, k = Phi.shape
    R = np.eye(k)
    
    for _ in range(q):
        Lambda = Phi @ R
        ### YOUR CODE HERE (3 lines): ###
        # 1. Compute SVD of: Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ diag(sum of Lambda**2))
        # 2. Update R = u @ vt
        # 3. (convergence check is done for you below)
        
        
        
        ### END YOUR CODE ###
        if np.max(np.abs(R_new - R)) < tol:
            break
        R = R_new
    
    return Phi @ R

print("Varimax function defined ✓")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
u, s, vt = np.linalg.svd(
    Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0)))
)
R_new = u @ vt
```
</details>

### 🟢 Exercise 5.2: Apply Varimax and plot (provided)

In [ ]:
# Exercise 5.2 (provided): Apply Varimax and plot heatmaps

# Healthy: rotate 3 PCs
L_h_raw = pca_h.components_[:3].T   # (6 × 3)
L_h_rot = varimax(L_h_raw).T        # rotate → (3 × 6)
for i in range(3):
    L_h_rot[i] /= np.linalg.norm(L_h_rot[i])
    if L_h_rot[i, np.argmax(np.abs(L_h_rot[i]))] < 0:
        L_h_rot[i] *= -1

# Impaired: rotate 3 PCs (to show PC3 is noise)
L_i_raw = pca_i.components_[:3].T
L_i_rot = varimax(L_i_raw).T
for i in range(3):
    L_i_rot[i] /= np.linalg.norm(L_i_rot[i])
    if L_i_rot[i, np.argmax(np.abs(L_i_rot[i]))] < 0:
        L_i_rot[i] *= -1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
iv = pca_i.explained_variance_ratio_

ax = axes[0]
im = ax.imshow(L_h_rot, cmap='RdBu_r', aspect='auto', vmin=-0.8, vmax=0.8)
ax.set_xticks(range(6)); ax.set_xticklabels(muscle_names, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels(['Syn 1 (Ext/Reach)', 'Syn 2 (Flex/Pull)', 'Syn 3 (Elbow Mod.)'], fontsize=9)
ax.set_title('Healthy: 3 Synergies (Varimax)', fontweight='bold')
for i in range(3):
    for j in range(6):
        ax.text(j, i, f'{L_h_rot[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(L_h_rot[i,j]) > 0.35 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Loading')

ax = axes[1]
im = ax.imshow(L_i_rot, cmap='RdBu_r', aspect='auto', vmin=-0.8, vmax=0.8)
ax.set_xticks(range(6)); ax.set_xticklabels(muscle_names, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels([f'Syn 1 ({iv[0]*100:.0f}%)', f'Syn 2 ({iv[1]*100:.0f}%)',
                    f'Syn 3 ({iv[2]*100:.1f}%)'], fontsize=9)
ax.set_title('Impaired: 3rd Synergy Lost', fontweight='bold')
for i in range(3):
    for j in range(6):
        ax.text(j, i, f'{L_i_rot[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(L_i_rot[i,j]) > 0.35 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Loading')
fig.suptitle('PCA Loadings After Varimax Rotation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 🟡 Exercise 5.3: Compare unrotated loadings
Plot the *unrotated* PCA loadings as a heatmap. Compare with the Varimax-rotated version above.
What differences do you see? Which is easier to interpret as muscle groupings?

In [ ]:
# Exercise 5.3: Unrotated loadings comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Healthy unrotated
L_h_unrot = pca_h.components_[:3].copy()
for i in range(3):
    if L_h_unrot[i, np.argmax(np.abs(L_h_unrot[i]))] < 0:
        L_h_unrot[i] *= -1

ax = axes[0]
hv = pca_h.explained_variance_ratio_
### YOUR CODE HERE: create the heatmap (imshow, set ticks, add text values) ###
# Hint: same pattern as the provided code above, but use L_h_unrot and label as PC1/PC2/PC3




### END YOUR CODE ###

# Impaired unrotated
L_i_unrot = pca_i.components_[:3].copy()
for i in range(3):
    if L_i_unrot[i, np.argmax(np.abs(L_i_unrot[i]))] < 0:
        L_i_unrot[i] *= -1

ax = axes[1]
im = ax.imshow(L_i_unrot, cmap='RdBu_r', aspect='auto', vmin=-0.8, vmax=0.8)
ax.set_xticks(range(6)); ax.set_xticklabels(muscle_names, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels([f'PC1 ({iv[0]*100:.0f}%)', f'PC2 ({iv[1]*100:.0f}%)',
                    f'PC3 ({iv[2]*100:.1f}%)'], fontsize=9)
ax.set_title('Impaired: UNROTATED', fontweight='bold')
for i in range(3):
    for j in range(6):
        ax.text(j, i, f'{L_i_unrot[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(L_i_unrot[i,j]) > 0.35 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Loading')

fig.suptitle('Without Varimax: PCs Ordered by Variance, Not Muscle Grouping',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
im = ax.imshow(L_h_unrot, cmap='RdBu_r', aspect='auto', vmin=-0.8, vmax=0.8)
ax.set_xticks(range(6)); ax.set_xticklabels(muscle_names, fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels([f'PC1 ({hv[0]*100:.0f}%)', f'PC2 ({hv[1]*100:.0f}%)', f'PC3 ({hv[2]*100:.0f}%)'])
ax.set_title('Healthy: UNROTATED', fontweight='bold')
for i in range(3):
    for j in range(6):
        ax.text(j, i, f'{L_h_unrot[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(L_h_unrot[i,j]) > 0.35 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Loading')
```
</details>

---
## 🟡 Part 6: PC Score Scatter

### Exercise 6.1: Scatter plot colored by target direction
Plot PC1 vs PC2 scores for healthy and impaired, coloring each point by target angle.

In [ ]:
# Exercise 6.1: PC score scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cmap = plt.cm.hsv
target_colors = [cmap(i / n_targets) for i in range(n_targets)]
targets_h = targets[healthy_mask]
targets_i = targets[impaired_mask]

for ax, (sc, tgt, label) in zip(axes,
        [(scores_h, targets_h, 'Healthy'), (scores_i, targets_i, 'Impaired')]):
    for ti in range(n_targets):
        mask = tgt == ti
        ### YOUR CODE HERE (1 line): scatter sc[mask, 0] vs sc[mask, 1] ###
        
        ### END YOUR CODE ###
    ax.set_xlabel('PC1 Score'); ax.set_ylabel('PC2 Score')
    ax.set_title(f'{label}', fontweight='bold')
    ax.legend(fontsize=7, ncol=2, title='Target'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
ax.scatter(sc[mask, 0], sc[mask, 1], c=[target_colors[ti]],
           label=f'{int(np.rad2deg(target_angles[ti]))}°', alpha=0.6, s=30)
```
</details>

---
## 🟠 Part 7: Reconstruction Error

### Exercise 7.1: Reconstruct from k PCs and plot MSE
For k = 1 to 6 PCs, reconstruct the data and compute mean squared error.

In [ ]:
# Exercise 7.1: Reconstruction error
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

for X_std, pca_obj, label, color in [
        (X_h, pca_h, 'Healthy', BLUE), (X_i, pca_i, 'Impaired', RED)]:
    errors = []
    for k in range(1, 7):
        ### YOUR CODE HERE (3 lines): ###
        # 1. Transform to k-dimensional scores: Z_k = pca_obj.transform(X_std)[:, :k]
        # 2. Reconstruct: X_recon = Z_k @ pca_obj.components_[:k]
        # 3. Compute MSE: mse = np.mean((X_std - X_recon) ** 2)
        
        
        
        ### END YOUR CODE ###
        errors.append(mse)
    ax.plot(range(1, 7), errors, 'o-', color=color, lw=2, ms=8, label=label)

ax.set_xlabel('Number of PCs'); ax.set_ylabel('MSE')
ax.set_title('Reconstruction Error', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xticks(range(1, 7))
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
Z_k = pca_obj.transform(X_std)[:, :k]
V_k = pca_obj.components_[:k]
X_recon = Z_k @ V_k
mse = np.mean((X_std - X_recon) ** 2)
```
</details>

---
## 🟡 Part 8: sklearn Pipeline

### Exercise 8.1: Chain StandardScaler + PCA in a Pipeline

In [ ]:
# Exercise 8.1: sklearn Pipeline
from sklearn.pipeline import Pipeline

### YOUR CODE HERE (3 lines): ###
# 1. Create Pipeline with ('scaler', StandardScaler()) and ('pca', PCA(n_components=3))
# 2. Fit-transform on healthy raw data: X_raw[healthy_mask]
# 3. Print the variance explained


### END YOUR CODE ###
print(f"Input shape: {X_raw[healthy_mask].shape}")
print(f"Output shape: {scores_pipe.shape}")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
pipe = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=3))])
scores_pipe = pipe.fit_transform(X_raw[healthy_mask])
print(pipe.named_steps['pca'].explained_variance_ratio_.round(3))
```
</details>

---
## 🟡 Part 8b: Looking Ahead — All Trials in PCA Space

So far we've looked at healthy and impaired separately. Now let's project **all 480 trials** into one shared PCA space.

### Exercise 8b.1: Plot all trials colored by target direction

Standardize all 480 trials together, fit PCA on the combined data, and scatter PC1 vs PC2 colored by target direction.

**Think about:** Do trials from both groups appear in each directional cluster? What does this suggest about what a classifier would need to do to tell the groups apart?

In [ ]:
# Exercise 8b.1: All 480 trials in shared PCA space

# Step 1: Standardize ALL trials (not just healthy)
scaler_all = StandardScaler()
X_all_std = scaler_all.fit_transform(X_raw)

# Step 2: Fit PCA on combined data
pca_all = PCA(n_components=2).fit(X_all_std)
sc_all = pca_all.transform(X_all_std)

# Step 3: Scatter plot colored by target direction
fig, ax = plt.subplots(figsize=(7, 5.5))
cmap = plt.cm.hsv

### YOUR CODE HERE ###
# Loop over the 8 target directions
# For each direction, scatter sc_all[mask, 0] vs sc_all[mask, 1]
# Color by cmap(ti / n_targets), label by degree


ax.set_xlabel('PC1 Score'); ax.set_ylabel('PC2 Score')
ax.set_title('All 480 Trials in PCA Space — Colored by Target Direction', fontweight='bold')
ax.legend(fontsize=8, ncol=4, title='Target', title_fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🟠 Part 9: Interpretation Questions

Answer these in your own words (in a new markdown cell or as comments):

**Q1.** Why does PCA on healthy data require 3 PCs for >90% variance, but impaired only needs 2?

**Q2.** What does Varimax rotation change, and what does it preserve?

**Q3.** A loading of +0.8 on Sh.H.Ext and −0.3 on Sh.H.Flex in the same synergy means what physiologically?

**Q4.** If a patient's scree plot needed only 1 PC for 95% variance, what would that suggest clinically?

---
## 🔴 Part 10 (Challenge): Cross-Validated Dimensionality Selection

Instead of an arbitrary 90% variance threshold, use leave-one-subject-out cross-validation:
1. Hold out one subject
2. Fit PCA on the remaining subjects
3. Reconstruct the held-out subject's data
4. Measure reconstruction error
5. Plot mean CV error vs. number of PCs

Which k gives the best generalization?

In [ ]:
# Exercise 10.1: Leave-one-subject-out CV for dimensionality selection
from sklearn.model_selection import LeaveOneGroupOut

# Subject IDs for healthy group
subject_ids_h = np.repeat(np.arange(n_healthy), n_targets * len(speed_durations))
logo = LeaveOneGroupOut()
max_k = 6

### YOUR CODE HERE ###
# 1. Loop over CV folds: for fold, (train_idx, test_idx) in enumerate(logo.split(...))
# 2. For each fold, standardize train/test
# 3. For k in 1..max_k, fit PCA(n_components=k) on train, reconstruct test, compute MSE
# 4. Store errors in a (n_folds × max_k) array
# 5. Plot mean ± std error vs k



### END YOUR CODE ###

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
cv_errors = np.zeros((logo.get_n_splits(X_h, groups=subject_ids_h), max_k))

for fold, (train_idx, test_idx) in enumerate(logo.split(X_h, groups=subject_ids_h)):
    scaler_cv = StandardScaler()
    X_train = scaler_cv.fit_transform(X_h[train_idx])
    X_test = scaler_cv.transform(X_h[test_idx])
    for k in range(1, max_k + 1):
        pca_cv = PCA(n_components=k).fit(X_train)
        X_test_recon = pca_cv.inverse_transform(pca_cv.transform(X_test))
        cv_errors[fold, k-1] = np.mean((X_test - X_test_recon) ** 2)
```
</details>

---
## 🎯 Lab Summary

| Concept | What You Did |
|---|---|
| 2-DOF kinematics | Forward kinematics: joint angles → hand position |
| Synergy model | Weight matrix W maps synergies → muscles |
| EMG processing | Rectification → low-pass envelope → peak amplitude |
| StandardScaler | Zero mean, unit variance per muscle |
| PCA & scree plot | Identify how many independent modules explain the data |
| Varimax rotation | Rotate PCs for physiological interpretability |
| Clinical insight | Fewer modules = less independent control (Clark et al.) |
| sklearn Pipeline | Chain preprocessing + PCA in reproducible code |
| Cross-validation | Data-driven dimensionality selection |

**Next week:** Classification — can we predict healthy vs. impaired from synergy features?

---
## Save Dataset for Week 5

Save the raw data and PCA results so Week 5 can load them directly.

In [ ]:
# Save dataset for Week 5
import pickle

week4_data = {
    'X_raw': X_raw,
    'labels': labels,
    'targets': targets,
    'subjects': np.array([si for si in range(20) for _ in range(24)]),
    'muscle_names': muscle_names,
    'target_angles': target_angles,
    'n_targets': n_targets,
    'sc_all': sc_all,
    'pca_all': pca_all,
    'scaler_all': scaler_all,
}

with open('week4_data.pkl', 'wb') as f:
    pickle.dump(week4_data, f)

print('Saved week4_data.pkl — ready for Week 5!')